# 03 - Results Analysis

Loads `screening_results.csv` / `summary.json` produced by
`crypticip screen`, ranks candidates, and plots score distributions,
tier counts, and a feature heatmap.

All cells gracefully no-op when the expected files are missing, so
you can run this notebook even if you haven't completed a real
screening run yet.

## 1. Setup

In [ ]:
import os, subprocess, pathlib
REPO_URL = 'https://github.com/Tommaso-R-Marena/cryptic-ip-pipeline.git'
WORK = pathlib.Path('/content/cryptic-ip-pipeline')
if not WORK.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORK)], check=True)
os.chdir(WORK)
!pip install -q -e . pandas matplotlib seaborn
import pandas as pd, numpy as np, json, pathlib
import matplotlib.pyplot as plt


## 2. Locate screening output

In [ ]:
ORGANISM = 'yeast'  # change to 'human' or 'dictyostelium' as needed
screen_dir = pathlib.Path(f'results/screening/{ORGANISM}')
csv_path = screen_dir / 'screening_results.csv'
top_path = screen_dir / 'screening_top.csv'
summary_path = screen_dir / 'summary.json'
for p in (csv_path, top_path, summary_path):
    print(f'{p}: {"OK" if p.exists() else "MISSING"}')


## 3. Load + preview

In [ ]:
if not csv_path.exists():
    print('No screening_results.csv. Run `crypticip screen --organism', ORGANISM, '` first.')
    df = pd.DataFrame()
else:
    df = pd.read_csv(csv_path)
    print('rows:', len(df), 'cols:', len(df.columns))
    display(df.head())


## 4. Rank candidates by composite score

In [ ]:
if len(df) and 'score' in df.columns:
    ranked = df.sort_values('score', ascending=False).reset_index(drop=True)
    cols = [c for c in ['accession','score','tier','volume_A3','depth_A',
                        'potential_kT_e','basic_count_5A','mean_plddt']
            if c in ranked.columns]
    display(ranked[cols].head(25))
else:
    print('No score column or empty results.')


## 5. Score distribution

In [ ]:
if len(df) and 'score' in df.columns:
    fig, ax = plt.subplots(figsize=(6,4))
    df['score'].dropna().plot.hist(bins=40, ax=ax)
    ax.set_xlabel('composite score'); ax.set_ylabel('count')
    ax.set_title(f'{ORGANISM}: screening score distribution (n={df["score"].notna().sum()})')
    plt.tight_layout(); plt.show()
else:
    print('Skipping score histogram — no data.')


## 6. Tier counts

In [ ]:
if len(df) and 'tier' in df.columns:
    counts = df['tier'].value_counts().reindex(['tier1','tier2','tier3','tier4']).fillna(0).astype(int)
    print(counts)
    fig, ax = plt.subplots(figsize=(5,3))
    counts.plot.bar(ax=ax, color=['#1f77b4','#2ca02c','#ff7f0e','#7f7f7f'])
    ax.set_title(f'{ORGANISM} tiers'); ax.set_ylabel('count')
    plt.tight_layout(); plt.show()
else:
    print('No tier column — nothing to plot.')


## 7. Feature heatmap for top 25 candidates

In [ ]:
if len(df) >= 5 and 'score' in df.columns:
    feats = [c for c in ['volume_A3','depth_A','potential_kT_e',
                         'basic_count_5A','basic_count_8A','basic_count_10A',
                         'mean_plddt','inv_sasa','volume_fit']
             if c in df.columns]
    if feats:
        top = df.sort_values('score', ascending=False).head(25)
        mat = top[feats]
        # z-score each column for visibility
        z = (mat - mat.mean()) / (mat.std(ddof=0).replace(0, np.nan))
        fig, ax = plt.subplots(figsize=(8, max(3, 0.3*len(top))))
        im = ax.imshow(z.values, aspect='auto', cmap='RdBu_r', vmin=-2.5, vmax=2.5)
        ax.set_xticks(range(len(feats))); ax.set_xticklabels(feats, rotation=40, ha='right')
        labels = top['accession'] if 'accession' in top.columns else top.index.astype(str)
        ax.set_yticks(range(len(top))); ax.set_yticklabels(labels, fontsize=7)
        ax.set_title(f'{ORGANISM} top 25 — feature z-scores')
        fig.colorbar(im, ax=ax, shrink=0.7)
        plt.tight_layout(); plt.show()
    else:
        print('No feature columns found.')
else:
    print('Not enough rows for a heatmap.')


## 8. Export ranked CSV

In [ ]:
OUT_DIR = pathlib.Path('results/reports/analysis'); OUT_DIR.mkdir(parents=True, exist_ok=True)
if len(df) and 'score' in df.columns:
    out = OUT_DIR / f'{ORGANISM}_ranked.csv'
    df.sort_values('score', ascending=False).to_csv(out, index=False)
    print('wrote', out)
else:
    print('Nothing to export.')


## 9. Summary

- This notebook is purely analytic — it doesn't re-run any expensive
  step. You can iterate on plots without touching the pipeline.
- All plots are made from your own `screening_results.csv`. There
  are no fabricated numbers anywhere in this notebook.
